<a href="https://colab.research.google.com/github/davidsjohnson/wise24_xai_ac/blob/main/notebooks/tutorial_image_xai_task4_crp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explaining Facial Expression Recognition
# Notebook 1:  XAI for Affective Computing (SoSe2024)

In this notebook you will attempt to generate explanations for predictions of a facial expression recognition (FER) CNN trained on raw image data, using a subset of the [AffectNet dataset](http://mohammadmahoor.com/affectnet/).

We will use a concept-based approach to generating explanations in this notebook. To do this we will use Concept Relevance Propagation (CRP), which we learned about in the paper ["From attribution maps to human-understandable explanations through Concept Relevance Propagation".](https://www.nature.com/articles/s42256-023-00711-8)

The documentation is still limited but the [CRP GitHub Repo](https://github.com/rachtibat/zennit-crp) has enough to get use started.  So make sure to review the README.  



To use this notebook, please make sure to go step by step through each of the cells review the code and comments along the way.

***NOTE**: This notebook runtime could be improved by using a GPU if available.*

## Notebook Setup

Make sure to uncomment code based on if you are running locally or via Google Colab

In [1]:
## uncomment and run this cell to use Google Colab
!git clone https://github.com/davidsjohnson/wise24_xai_ac.git
# fix xgboost incompatiblity issue
%pip uninstall -y -q scikit-learn
%pip install -q scikit-learn==1.5.2

Cloning into 'wise24_xai_ac'...
remote: Enumerating objects: 506, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 506 (delta 21), reused 16 (delta 5), pack-reused 460 (from 1)
Receiving objects: 100% (506/506), 104.95 MiB | 10.62 MiB/s, done.
Resolving deltas: 100% (81/81), done.
Updating files: 100% (399/399), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 115.2 MB/s eta 0:00:00


In [2]:
import sys
import os
# sys.path.append(os.path.realpath('../'))  # uncomment this line if you are running this notebook locally
sys.path.append(os.path.realpath('wise24_xai_ac')) # uncomment this line if you are running this notebook on Google Colab

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import randint, uniform

import torch
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F
import torch.nn as nn

import seaborn as sns
import matplotlib.pyplot as plt

from skimage import io

import utils
import img_utils
import models
import evaluate

In [4]:
colab = True # change to False if running locally
base_dir = Path('../data/') if not colab else Path('wise24_xai_ac/data/')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## XAI for FER with  Convoluational Neural Nets

### Setup the Pytorch Data Loader

In [5]:
#class labels
class_names = ['Neutral', 'Happy', 'Sad', 'Surprise', 'Fear', 'Disgust', 'Anger', 'Contempt']

# Setup XAI Data from AffectNet Deep Learning Model
TRAIN_MEAN = [0.485, 0.456, 0.406]
TRAIN_STD = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD),
    transforms.Resize((224, 224))
])

data_dir = base_dir / 'affectnet/val_class'
dataset = datasets.ImageFolder(root=data_dir, transform=test_transform)
images = [io.imread(f[0]) for f in dataset.imgs]
y_true = np.array([f[1] for f in dataset.imgs])
y_labels = [class_names[f[1]] for f in dataset.imgs]
dataloader = DataLoader(dataset, batch_size=80, shuffle=False)

### Load Pretrained Model

In [6]:
# download checkpoint
ckpt_link = 'https://uni-bielefeld.sciebo.de/s/0tAa2wPhGxSDjbM/download'
ckpt_path = utils.download_file(ckpt_link,
                                'affectnet.pth',
                                cache_dir= base_dir / 'affectnet/model',
                                extract=False,
                                force_download=False
                                )
ckpt_path

File already exists at: wise24_xai_ac/data/affectnet/model/affectnet.pth


PosixPath('wise24_xai_ac/data/affectnet/model/affectnet.pth')

In [7]:
model = models.ResNet18(n_classes=len(class_names), pretrained=True)
model.to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval();

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 82.8MB/s]
<ipython-input-7-c89b7f1c43de>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer b

### Evaluation of Model

This model performs much better than the AU dataset, but still with only around $60\%$ accuracy.  But this is pretty close the state-of-the-art for the AffectNet dataset

In [8]:
inverse_weights = torch.from_numpy(1.0/np.array([74874, 134415, 25459, 14090, 6378, 3803, 24882, 3750])).type(torch.float32).to(device)
loss = torch.nn.CrossEntropyLoss(weight=inverse_weights)
_, _, y_preds, probs = evaluate.evaluate_model(model, dataloader, loss, device=device)

y_preds = np.array(y_preds)
# validate predictions and true values
(y_preds == y_true).mean()

Evaluation Loss: 1.3183, Evaluation Accuracy: 0.5875


0.5875

## Generate Explanations

In [11]:
!pip install -q zennit-crp[fast_img]

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00


In [12]:
from crp.attribution import CondAttribution
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names

from zennit.composites import EpsilonPlusFlat
from zennit.canonizers import SequentialMergeBatchNorm

from crp.visualization import FeatureVisualization
from crp.image import plot_grid, imgify

### Task 1 - Generate CRP Attribution Maps

[CRP GitHub Repo](https://github.com/rachtibat/zennit-crp)

Review the [Attributions Tutorial](https://github.com/rachtibat/zennit-crp/blob/master/tutorials/attributions.ipynb) for info on CRP and help with these tasks

**Task 1.1:** Generate a basic feature attribution map for one example from the test data.  The feature attribution should be conditioned on just the predicted class.  This will provide us with a standard saliency map similar other approaches like LRP or Integrated gradients.


**Task1.2:** Generate attribution maps for three randomly selected "concepts" from the last layer of the network.  You can use the `get_layer_names` function to find the name of the last layer of our model.  Each "concept" defined as individual feature map from that layer. In our model the last layer has 512 feature maps, so just choose 3 random feature maps to use in your conditions for generating attribution values.  Then visualize the attibution maps.


You can use the "Broadcast" functionality described in the tutorial to do this.

Now the generated attribution maps represent the pixels of the image most important to that specific feature map.  (But note, that we do not yet now how important these feature maps are since we just randomly selected them)

**Task 1.3:** Identify the top 5 concepts (i.e. feature maps) from the last layer of the network for your selected image's predicted class.  Then plot their corresponding feature maps.  

**Preview the Dataset with Predictions**

The code below will display images from the XAI dataset.
- Try changing value of `start` to get a new set of images (there are 10 images for each class; for example, the class happy will be at indexes 10-19)
- Search through the images to find some that might be interesting to Explain

In [ ]:
start=40
img_utils.display_nine_images(images, y_true, y_preds, start)

In [13]:
####### Select Your images #######
##################################

idx =
cls =

In [ ]:
# get sample
sample = images[idx]
sample = test_transform(sample)
sample = sample.unsqueeze(0)
sample = sample.to(device)
imgify(sample[0])

In [15]:
###### Enter your Code Below ######
##################################




### Task 2 - Relevance Maximization

Relvance Maximation aims to identify the top images that maximize the relevence score for a given concept. The idea is to find a subsample of the dataset that helps to visually understand what is the human-understandable concept the model learned for that model concept.  

The [Feature Visualization Tutorial Notebook](https://github.com/rachtibat/zennit-crp/blob/master/tutorials/feature_visualization.ipynb) will help you with this task.  

**Tasks 2.1:** Using the previously identified top 10 concept ids, use the `FeatureVisualization` class to find the images that maximize the relevance values of each concept. Then plot the images using the `plot_grid` function.  


**Task 2.2:** Answer Question 1 Below.

In [16]:
###### Enter your Code Below ######
##################################


**Question 1**

**Q 1.1**  
Try to semantically describe the identified concepts based on the images selected via RelMax.  

**Q 1.2**   
Is it as easy and straightforword to describe the concepts as they suggest in the original paper?

**Q 1.3:**  
How do these results compare with teh saliency maps from SHAP and Integrated Gradients?

**Q 1.4: (Optional)**  
Can you think of how you might use CRP and RelMax to get a more detailed understanding of the different layers in the network?

Write your answer here...